# Archived investigation: semester cycles and occupancy

This notebook preserves the earlier semester-phase, weekly-drop, and daily-noon investigation for traceability. It is archived and is not part of the active investigation. Its historical reports are in `derived/archive/semester_cycle_occupancy_investigation/reports/`.

The active investigation now studies timestamp-aligned environmental measurements and occupancy within the May-December 2024 period.


# Occupancy investigation design

> The preprocessing cell above is deliberately unchanged. This section turns the proposed ideas into a complete, testable investigation plan. Hypotheses are not findings: verify dates, cadence, timezone, and academic-calendar phases before interpretation.

## Engineering problem, aim, and intended use

University spaces are used unevenly over academic weeks and within each day. Facilities planners could use a better description of when monitored spaces are busy to inform staffing, cleaning, or future space-planning decisions. The project will test whether academic-calendar phase helps explain or predict recorded occupancy, and describe weekly changes and daily peaks. It does not measure total campus use, room capacity, learning outcomes, or causal effects of exams on attendance.

**Aim:** assess whether university semester phase and time-of-week patterns are useful predictors of recorded occupancy in the five monitored floorspaces during the dates covered by the data.

## Hypotheses and measurable targets

- **H1 - Semester cycle:** occupancy is lower during official university breaks than teaching periods; occupancy may also differ during the first three teaching weeks and official assessment/exam periods.
- **H2 - Weekly changes:** week-to-week occupancy changes are structured rather than random, with identifiable drops in particular weeks, potentially around breaks or transitions.
- **H3 - Daily peak:** the typical daily occupancy peak occurs around noon; test whether this holds across spaces and weekdays, rather than relying on an overall average.

The outcome is **recorded headcount**, not verified unique people. Candidate units are a fixed local-time bin, day, or week. Fix the unit, missing-data rule, coverage threshold, and whether the summary is row-weighted or time-weighted only after the cadence audit. For H1, the null is no phase difference after accounting for weekday and time-of-day composition. For H2, use persistence/no systematic change as the baseline. For H3, define a noon window before viewing peak results (initial proposal: 11:00-13:00 local time, with sensitivity analysis). Report effect sizes and uncertainty, not just significance; any hypothesis may be unsupported.

## Scope and data decisions

Analyze the five floorspaces separately first: the occupancy-location lookup is known to be unreliable, so do not join environmental measurements. Use only the observed date range. Verify timezone and complete coverage before assigning academic phases. Use the official university calendar for every included year (initially focus on 2024; include later years only after their dates are verified) for teaching weeks, breaks, and exam periods; mid-semester tests are course-specific and should be included only when their dates are independently available. Document the calendar source, version, and every boundary; never infer phases from occupancy values. The current generated streams have uneven coverage: their observed UTC dates span 2023-11-23 to 2026-04-22, but one floorspace ends in 2024 while others continue into 2026. Compare phases within floorspace and compare years only where calendar-phase coverage overlaps.

The first analytical question is what an occupancy row means. If observations are regularly sampled, a fixed-bin mean may be reasonable. If event-driven or irregular, a row average overweights densely sampled periods; consider a defensible time-weighted or occupancy-duration estimate instead. Do not assume a headcount persists between events until the sensor semantics support that assumption.

## Increasing-scale investigation sequence

1. **Resolve the data:** verify date coverage, missingness, timestamp offsets, chronological order, cadence and gaps, and headcount validity, cadence and gaps, and headcount range by floorspace. Decide the local timezone and aggregation after this audit.
2. **H1:** align comparable weekdays/time bins to official semester phases; estimate absolute and relative contrasts by space. Compare a weekday/time-only baseline with the same model plus phase. Discuss whether phase adds useful predictive information, not merely a statistically significant coefficient.
3. **H2:** form an agreed weekly measure, plot the sequence, and calculate absolute and percentage week-over-week changes. Define a material-drop rule before identifying affected weeks. Check alignment with breaks/exams. Compare next-week persistence with a parsimonious calendar-aware forecast.
4. **H3:** form justified daily profiles, find each valid day-space peak bin, and quantify the fraction of peaks in the noon window. Compare weekday/weekend and floorspace patterns; report day coverage and sensitivity to bin width/window.
5. **Synthesize:** answer each hypothesis separately, explain practical importance without causal overclaim, and state limits on generalization.

## Models, alternatives, and validation

Begin with transparent plots/summaries and simple baselines. H1 can use a regression-style phase comparison controlling for time-of-week and floorspace, with uncertainty clustered/bootstrapped at days or weeks rather than individual autocorrelated rows. H2's baseline is next week's value equals this week's value; compare a small calendar-aware model only if there are enough independent weeks. H3 is initially descriptive; make it predictive only if an operational forecasting use is justified. Keep all floorspaces separate before any pooled result. Defer complex models and causal claims: unequal multi-year coverage cannot establish a recurring pattern unless comparable phases are observed in multiple years for the same floorspace; calendar association is not causation.

Keep observations in time order: fit/tune on earlier dates and evaluate on later dates. Never randomly split neighboring time-series rows. Compare forecasts with persistence using MAE and a scale-aware metric; report by floorspace/phase where sample sizes permit. Check residuals, coverage, missingness, and sensitivity to aggregation, incomplete-day rules, and phase boundaries. If temporal coverage is too short for a meaningful holdout, report descriptive evidence and explicitly withhold predictive-performance claims.

## Project-report evidence and reproducibility

The completed investigation should present: engineering problem and intended use; defined target and computational method; justified preprocessing/aggregation; credible alternatives and why rejected; validation and baseline comparison; interpretable plots/tables and effect sizes; practical implications; and limitations/lessons. Record input paths and preprocessing validation, calendar citation, timezone, unit/bin width, thresholds, exclusions, model formula, baseline, chronological train/test dates, metrics, uncertainty method, and software versions. Discuss unknown space capacity/purpose, sensor coverage/accuracy, calendar incompleteness, autocorrelation, unequal multi-year coverage and limited repeated phase observations, and the distinction between recorded and actual occupancy.

**Decision gate:** do not claim results for H1-H3 until the diagnostic below establishes actual dates, cadence, ordering, and timezone. Calendar labels and analysis parameters are explicit inputs, not hidden assumptions.

## 0. Data audit: resolve temporal meaning before testing hypotheses

Run this audit after preprocessing. It streams each generated floorspace file and reports date coverage, missing/invalid headcounts, timestamp timezone-offset completeness, ordering problems, and observed positive inter-record intervals. The input examples include explicit UTC offsets; the code refuses to silently interpret naive timestamps as local time. Review whether intervals support a regular-sampling interpretation, and confirm sensor semantics before treating a value as lasting until the next record. The first run on the current derived outputs found records from 2023-11-23 through 2026-04-22 UTC, uneven end dates by floorspace, explicit timezone offsets on all rows, and no missing/invalid timestamps or headcounts. All five streams contain out-of-order rows (4 to 1,317 per stream). Rerun after each preprocessing update; do not treat file-order interval quantiles as a final cadence conclusion. This audit creates no derived analysis files.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next((parent for parent in Path.cwd().resolve().parents if (parent / "pre-Processing.ipynb").is_file()), Path.cwd().resolve())
STREAM_DIR = ROOT / "derived" / "occupancy_streams_by_floorspace"
MANIFESTS = {"stream_row_counts.csv", "occupancystatuschangedate_audit.csv"}
STREAM_FILES = sorted(path for path in STREAM_DIR.glob("*.csv") if path.name not in MANIFESTS)
if not STREAM_FILES:
    raise FileNotFoundError(f"No processed occupancy streams found in {STREAM_DIR}; run the preprocessing cell first.")

# Check timestamps in bounded chunks so this audit does not load every event into memory.
audit_rows = []
for path in STREAM_FILES:
    n_rows = n_missing_count = n_invalid_count = n_invalid_time = n_missing_offset = n_out_of_order = 0
    first_utc = last_utc = None
    prior_utc = None
    interval_parts = []
    for chunk in pd.read_csv(path, dtype=str, keep_default_na=False, chunksize=250_000):
        n_rows += len(chunk)
        raw_time = chunk["collecteddate"].astype("string").str.strip()
        has_clock = raw_time.str.contains(r"\d{2}:\d{2}", regex=True, na=False)
        has_offset = raw_time.str.contains(r"(?:Z|[+-]\d{2}:?\d{2}|[+-]\d{2})\s*$", case=False, regex=True, na=False)
        n_missing_offset += int((~(has_clock & has_offset)).sum())
        times = pd.to_datetime(raw_time, format="mixed", errors="coerce", utc=True)
        n_invalid_time += int(times.isna().sum())
        valid_times = times.dropna()
        if not valid_times.empty:
            first_utc = valid_times.min() if first_utc is None else min(first_utc, valid_times.min())
            last_utc = valid_times.max() if last_utc is None else max(last_utc, valid_times.max())
        differences = times.diff()
        if prior_utc is not None and not times.empty and pd.notna(times.iloc[0]):
            differences.iloc[0] = times.iloc[0] - prior_utc
        n_out_of_order += int((differences < pd.Timedelta(0)).sum())
        positive = differences[differences > pd.Timedelta(0)].dt.total_seconds().dropna()
        if not positive.empty:
            interval_parts.append(positive.to_numpy(dtype="float64"))
        if not times.empty and pd.notna(times.iloc[-1]):
            prior_utc = times.iloc[-1]
        count_text = chunk["headcount"].astype("string").str.strip()
        missing_count = count_text.isna() | count_text.str.casefold().isin({"", "nan", "null", "none"})
        numeric_count = pd.to_numeric(count_text.mask(missing_count), errors="coerce")
        n_missing_count += int(missing_count.sum())
        n_invalid_count += int((~missing_count & numeric_count.isna()).sum())
    intervals = np.concatenate(interval_parts) if interval_parts else np.array([], dtype="float64")
    audit_rows.append({
        "floorspaceid": path.stem,
        "rows": n_rows,
        "coverage_start_utc": first_utc,
        "coverage_end_utc": last_utc,
        "missing_headcount": n_missing_count,
        "invalid_headcount": n_invalid_count,
        "invalid_timestamp": n_invalid_time,
        "timestamp_without_explicit_offset": n_missing_offset,
        "out_of_order_intervals": n_out_of_order,
        "median_positive_row_order_interval_seconds": float(np.median(intervals)) if intervals.size else np.nan,
        "p95_positive_row_order_interval_seconds": float(np.quantile(intervals, 0.95)) if intervals.size else np.nan,
    })
occupancy_audit = pd.DataFrame(audit_rows)
display(occupancy_audit)
if occupancy_audit["invalid_timestamp"].any() or occupancy_audit["invalid_headcount"].any():
    raise ValueError("Resolve invalid values before continuing to the hypothesis analyses.")
if occupancy_audit["timestamp_without_explicit_offset"].any():
    print("WARNING: Some timestamps lack an explicit timezone offset; do not derive local day/hour until their timezone is resolved.")
if occupancy_audit["out_of_order_intervals"].any():
    print("WARNING: Some rows are not chronological; the displayed row-order interval quantiles are provisional. Sort by timestamp and event ID before cadence-based aggregation.")
print("Coverage and cadence are now observable. Choose and document an aggregation unit and local timezone before continuing.")

,floorspaceid,rows,coverage_start_utc,coverage_end_utc,missing_headcount,invalid_headcount,invalid_timestamp,timestamp_without_explicit_offset,out_of_order_intervals,median_positive_row_order_interval_seconds,p95_positive_row_order_interval_seconds
0,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,2169461,2023-11-23 13:37:32+00:00,2026-04-22 13:02:18+00:00,0,0,0,0,1317,4.0,70.0
1,227f1068-9e41-469f-bef0-275f27a0bdba,1500394,2024-03-03 04:06:30+00:00,2026-04-22 12:09:42+00:00,0,0,0,0,5,4.0,100.0
2,3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8,1551583,2024-04-30 14:00:16+00:00,2024-12-30 12:01:56+00:00,0,0,0,0,4,2.0,30.0
3,941f0352-bb69-47d0-a9c5-ac29eb4f7319,1975423,2023-11-23 13:36:52+00:00,2026-04-22 13:02:38+00:00,0,0,0,0,802,4.0,74.0
4,cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b,1894871,2024-03-03 04:06:30+00:00,2026-04-22 12:54:50+00:00,0,0,0,0,6,4.0,40.0


Coverage and cadence are now observable. Choose and document an aggregation unit and local timezone before continuing.


## 1. H1 - Are semester phases associated with occupancy?

**Question:** are comparable occupancy summaries lower during official breaks than teaching periods, and do the first three teaching weeks or official exam periods differ?

**Required inputs:** first complete the cadence audit; verify timestamp offsets/local timezone; then create a calendar table with one row per local date and fields `date`, `phase`, and `teaching_week`. Cite the official calendar source and record any dates excluded from the observed coverage. Only add mid-semester assessment labels where relevant dates are known independently.

**Method:** use the aggregation justified by the sampling audit. Compare like-for-like weekdays and time-of-day bins by phase and floorspace. Report phase-specific sample/day coverage, absolute and relative differences, and uncertainty based on independent days/weeks. Fit a transparent baseline with weekday/time and floorspace controls, then compare the same specification with semester phase added. Treat a phase as useful for prediction only if it improves later-date holdout error over the baseline.

**Alternatives/checks:** report spaces separately before pooling; sensitivity-test incomplete days, bin width, phase boundaries, and row-weighted versus defensible time-weighted estimates. The calendar phase is a predictor, not proof of causation. Do not interpret phase estimates if the data cover too few comparable days or phases.

## 2. H2 - Which individual weeks show occupancy drops?

**Target:** one documented weekly occupancy measure per floorspace, constructed from the valid bins/days defined after the cadence audit. Use the university's documented week boundaries; identify partial weeks and show their coverage rather than comparing them as if complete.

**Analysis:** plot the weekly series by floorspace. Calculate absolute change `week_t - week_(t-1)` and percentage change only when the prior value is nonzero. Before naming low-use weeks, set a meaningful drop threshold (for example, a percentage plus a minimum absolute change) and state its rationale. Compare flagged weeks with independently sourced teaching/break/exam phases. Report the number of observed days and missingness in each weekly estimate.

**Prediction and validation:** for next-week occupancy, compare persistence (next week's value equals the current week's value) against a small model using only information available at forecast time, such as calendar phase and recent weeks. Use a chronological holdout and MAE plus a scale-aware metric; compare against the baseline. If there are too few complete weeks to hold out a meaningful test period, retain the descriptive week-change analysis and do not claim predictive skill. Sensitivity-test partial weeks, phase boundaries, and drop thresholds.

## 3. H3 - Do daily occupancy peaks occur around noon?

**Target:** the local-time bin with the highest justified occupancy estimate for each floorspace/day. Pre-register the initial noon window as 11:00-13:00 local time; also show sensitivity to a narrower window and to the bin width chosen from cadence evidence.

**Analysis:** plot mean/median daily profiles and the distribution of each valid day's peak time. Report the proportion of valid days whose peak is inside the noon window, with uncertainty calculated across days, not individual autocorrelated rows. Compare weekdays/weekends and floorspaces and show incomplete-day coverage. Do not declare a noon peak when multiple bins tie or day coverage is insufficient; document tie handling before calculation.

**Alternative and interpretation:** an overall mean-day curve is easy to communicate but can hide variation; pair it with daily peak-time distributions. If sampling is irregular, use a justified time-weighted profile rather than a row-weighted average. This is a descriptive hypothesis unless an operational forecast horizon is separately defined and validated. A noon peak in these instrumented spaces does not imply a campus-wide peak.

In [3]:
from datetime import timedelta
from pathlib import Path
import numpy as np
import pandas as pd

# Use the university's local civil time for academic dates and daily profiles.
TIMEZONE = "Australia/Melbourne"
DAYTIME_HOURS = list(range(8, 18))
NOON_HOURS = {11, 12}
MIN_COMPLETE_DAYS_PER_WEEK = 3
ROOT = next((candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (candidate / "pre-Processing.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this cell with the project repository as the working directory.")
STREAM_DIR = ROOT / "derived" / "occupancy_streams_by_floorspace"
OUTPUT_DIR = ROOT / "derived" / "archive" / "semester_cycle_occupancy_investigation" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS = {"stream_row_counts.csv", "occupancystatuschangedate_audit.csv"}
STREAM_FILES = sorted(path for path in STREAM_DIR.glob("*.csv") if path.name not in MANIFESTS)
if len(STREAM_FILES) != 5:
    raise ValueError(f"Expected five floorspace streams, found {len(STREAM_FILES)} in {STREAM_DIR}")

# Monash 2024 semester dates from the university's summary page archived 4 Feb 2024.
# Source: https://web.archive.org/web/20240204004333/https://www.monash.edu/students/admin/dates/summary-dates
# All ranges are inclusive local dates; the first-three-week labels use the Monday teaching-week starts.
CALENDAR = [
    ("2024-02-26", "2024-03-17", "Semester 1 teaching weeks 1-3"),
    ("2024-03-18", "2024-05-24", "Semester 1 teaching week 4+"),
    ("2024-04-01", "2024-04-05", "Semester 1 mid-semester break"),
    ("2024-05-27", "2024-05-31", "Semester 1 SWOTVAC"),
    ("2024-06-03", "2024-06-21", "Semester 1 final assessments"),
    ("2024-07-22", "2024-08-11", "Semester 2 teaching weeks 1-3"),
    ("2024-08-12", "2024-10-18", "Semester 2 teaching week 4+"),
    ("2024-09-23", "2024-09-27", "Semester 2 mid-semester break"),
    ("2024-10-21", "2024-10-25", "Semester 2 SWOTVAC"),
    ("2024-10-28", "2024-11-15", "Semester 2 final assessments"),
]

def calendar_phase(local_date):
    date_text = local_date.isoformat()
    # More specific break/exam intervals must override the enclosing teaching period.
    for start, end, label in CALENDAR[2:5] + CALENDAR[7:10]:
        if start <= date_text <= end:
            return label
    for start, end, label in (CALENDAR[0], CALENDAR[1], CALENDAR[5], CALENDAR[6]):
        if start <= date_text <= end:
            return label
    return "Outside listed teaching/assessment periods"

def load_and_sort(path):
    """Parse and stably order one bounded-to-one-floorspace stream by UTC time and source ID."""
    data = pd.read_csv(path, usecols=["id", "headcount", "collecteddate"],
                       dtype={"id": "string", "headcount": "string", "collecteddate": "string"},
                       keep_default_na=False)
    timestamp = pd.to_datetime(data["collecteddate"], format="mixed", errors="coerce", utc=True)
    if timestamp.isna().any():
        raise ValueError(f"{path.name}: {int(timestamp.isna().sum())} invalid timestamps")
    count_text = data["headcount"].astype("string").str.strip()
    missing = count_text.isna() | count_text.str.casefold().isin({"", "nan", "null", "none"})
    headcount = pd.to_numeric(count_text.mask(missing), errors="coerce")
    if headcount.isna().any() or not np.isfinite(headcount).all() or (headcount < 0).any() or (headcount % 1 != 0).any():
        raise ValueError(f"{path.name}: headcount is missing, non-finite, negative, or non-integer")
    data = data.assign(timestamp_utc=timestamp, headcount_numeric=headcount)
    inversions_before = int(data["timestamp_utc"].diff().lt(pd.Timedelta(0)).sum())
    duplicate_ids = int(data["id"].duplicated().sum())
    duplicate_times = int(data["timestamp_utc"].duplicated().sum())
    # Timestamp strings may have different UTC offsets; sort parsed instants, not source text.
    data = data.sort_values(["timestamp_utc", "id"], kind="stable").reset_index(drop=True)
    intervals = data["timestamp_utc"].diff().dt.total_seconds()
    if intervals.lt(0).any():
        raise AssertionError(f"{path.name}: chronological sort did not remove negative intervals")
    audit = {
        "floorspaceid": path.stem,
        "rows": len(data),
        "coverage_start_local": data["timestamp_utc"].iloc[0].tz_convert(TIMEZONE),
        "coverage_end_local": data["timestamp_utc"].iloc[-1].tz_convert(TIMEZONE),
        "out_of_order_rows_before_sort": inversions_before,
        "out_of_order_rows_after_sort": int(intervals.lt(0).sum()),
        "duplicate_source_ids": duplicate_ids,
        "duplicate_timestamps": duplicate_times,
        "median_positive_interval_seconds": intervals[intervals > 0].median(),
        "p95_positive_interval_seconds": intervals[intervals > 0].quantile(0.95),
        "gaps_over_15_minutes": int(intervals.gt(900).sum()),
    }
    return data, audit

# First pass verifies each stream and establishes the intersection where all five locations have records.
source_audits = []
coverage_bounds = []
for source_path in STREAM_FILES:
    sorted_data, audit = load_and_sort(source_path)
    source_audits.append(audit)
    coverage_bounds.append((sorted_data["timestamp_utc"].iloc[0], sorted_data["timestamp_utc"].iloc[-1]))
    del sorted_data
common_start_utc = max(start for start, _ in coverage_bounds)
common_end_utc = min(end for _, end in coverage_bounds)
if common_start_utc >= common_end_utc:
    raise ValueError("The five floorspaces have no shared timestamp interval.")
common_start_local = common_start_utc.tz_convert(TIMEZONE)
common_end_local = common_end_utc.tz_convert(TIMEZONE)
coverage_audit = pd.DataFrame(source_audits)
coverage_audit["common_window_start_local"] = common_start_local
coverage_audit["common_window_end_local"] = common_end_local
coverage_audit["analysis_note"] = "All comparative results restricted to shared five-space timestamp window; source files are unchanged."

# Second pass: sort, filter to shared coverage, and reduce records to local-day/local-hour evidence.
hourly_tables = []
for source_path in STREAM_FILES:
    data, _ = load_and_sort(source_path)
    data = data.loc[data["timestamp_utc"].between(common_start_utc, common_end_utc)].copy()
    local = data["timestamp_utc"].dt.tz_convert(TIMEZONE)
    data["local_date"] = local.dt.date
    data["local_hour"] = local.dt.hour
    data["weekday"] = local.dt.dayofweek
    data["phase"] = data["local_date"].map(calendar_phase)
    hourly = (data.groupby(["local_date", "local_hour", "weekday", "phase"], as_index=False)
              .agg(mean_headcount=("headcount_numeric", "mean"),
                   median_headcount=("headcount_numeric", "median"),
                   records=("headcount_numeric", "size")))
    hourly.insert(0, "floorspaceid", source_path.stem)
    hourly_tables.append(hourly)
    del data
hourly_evidence = pd.concat(hourly_tables, ignore_index=True)
daytime = hourly_evidence[hourly_evidence["local_hour"].isin(DAYTIME_HOURS)].copy()

# A complete analysis day has observations in every predeclared weekday business-hour bin (08:00-17:59).
coverage_by_day = (daytime.groupby(["floorspaceid", "local_date"])["local_hour"]
                   .nunique().rename("observed_daytime_hours").reset_index())
complete_days = coverage_by_day.loc[coverage_by_day["observed_daytime_hours"].eq(len(DAYTIME_HOURS)),
                                    ["floorspaceid", "local_date"]]
complete_daytime = daytime.merge(complete_days, on=["floorspaceid", "local_date"], how="inner")
complete_daytime = complete_daytime.sort_values(["floorspaceid", "local_date", "local_hour"])
daily_evidence = (complete_daytime.groupby(["floorspaceid", "local_date"], as_index=False)
                  .agg(daytime_mean_headcount=("mean_headcount", "mean"),
                       observed_daytime_hours=("local_hour", "nunique"),
                       source_records=("records", "sum")))
daily_evidence["phase"] = daily_evidence["local_date"].map(calendar_phase)
daily_evidence["week_start"] = daily_evidence["local_date"].map(
    lambda date: date - timedelta(days=date.weekday()))

# H1: show phase support and estimate matched weekday-by-hour descriptive contrasts, not causal effects.
phase_summary = (daytime.groupby(["floorspaceid", "phase"], as_index=False)
                 .agg(observed_dates=("local_date", "nunique"),
                      observed_day_hour_bins=("local_date", "size"),
                      mean_of_observed_day_hour_means=("mean_headcount", "mean"),
                      median_of_observed_day_hour_means=("mean_headcount", "median"),
                      source_records=("records", "sum")))
daytime["weekday_name"] = pd.to_datetime(daytime["local_date"].astype(str)).dt.day_name()
contrast_input = complete_daytime.copy()
strata = (contrast_input.groupby(["floorspaceid", "phase", "weekday", "local_hour"], as_index=False)
          .agg(stratum_mean=("mean_headcount", "mean"), stratum_days=("local_date", "nunique")))
contrast_pairs = [
    ("Semester 1 teaching week 4+", "Semester 1 mid-semester break"),
    ("Semester 1 teaching week 4+", "Semester 1 SWOTVAC"),
    ("Semester 1 teaching week 4+", "Semester 1 final assessments"),
    ("Semester 2 teaching week 4+", "Semester 2 teaching weeks 1-3"),
    ("Semester 2 teaching week 4+", "Semester 2 mid-semester break"),
    ("Semester 2 teaching week 4+", "Semester 2 SWOTVAC"),
    ("Semester 2 teaching week 4+", "Semester 2 final assessments"),
]
contrast_rows = []
for space in sorted(daytime["floorspaceid"].unique()):
    space_strata = strata[strata["floorspaceid"].eq(space)]
    for baseline_phase, comparison_phase in contrast_pairs:
        baseline = space_strata[space_strata["phase"].eq(baseline_phase)]
        comparison = space_strata[space_strata["phase"].eq(comparison_phase)]
        matched = baseline.merge(comparison, on=["weekday", "local_hour"], suffixes=("_baseline", "_comparison"))
        available_dates = phase_summary[(phase_summary["floorspaceid"].eq(space)) &
                                        phase_summary["phase"].isin([baseline_phase, comparison_phase])]
        date_counts = dict(zip(available_dates["phase"], available_dates["observed_dates"]))
        if matched.empty:
            base_mean = comparison_mean = difference = percent = np.nan
        else:
            base_mean = matched["stratum_mean_baseline"].mean()
            comparison_mean = matched["stratum_mean_comparison"].mean()
            difference = comparison_mean - base_mean
            percent = (difference / base_mean * 100) if base_mean else np.nan
        contrast_rows.append({
            "floorspaceid": space,
            "baseline_phase": baseline_phase,
            "comparison_phase": comparison_phase,
            "baseline_complete_days": int(daily_evidence.loc[(daily_evidence["floorspaceid"].eq(space)) &
                    daily_evidence["phase"].eq(baseline_phase), "local_date"].nunique()),
            "comparison_complete_days": int(daily_evidence.loc[(daily_evidence["floorspaceid"].eq(space)) &
                    daily_evidence["phase"].eq(comparison_phase), "local_date"].nunique()),
            "matched_weekday_hour_strata": len(matched),
            "matched_baseline_mean": base_mean,
            "matched_comparison_mean": comparison_mean,
            "matched_difference_headcount": difference,
            "matched_percent_change": percent,
            "interpretation": "Descriptive matched contrast only; not causal or a validated forecast."
        })
phase_contrasts = pd.DataFrame(contrast_rows)

# H2: weekly values use equal-weight means of complete 08:00-17:59 daily profiles.
weekly_rows = []
for space, group in daily_evidence.groupby("floorspaceid"):
    observed = (group.groupby("week_start", as_index=False)
                .agg(weekly_mean_headcount=("daytime_mean_headcount", "mean"),
                     complete_days=("local_date", "nunique")))
    first_week, last_week = min(observed["week_start"]), max(observed["week_start"])
    weeks = pd.DataFrame({"week_start": pd.date_range(first_week, last_week, freq="7D").date})
    observed = weeks.merge(observed, on="week_start", how="left").sort_values("week_start")
    observed["complete_days"] = observed["complete_days"].fillna(0).astype(int)
    observed["floorspaceid"] = space
    observed["week_start_phase"] = observed["week_start"].map(calendar_phase)
    eligible = observed["complete_days"].ge(MIN_COMPLETE_DAYS_PER_WEEK)
    previous_eligible = eligible.shift(1, fill_value=False)
    change = observed["weekly_mean_headcount"].diff()
    percent = observed["weekly_mean_headcount"].pct_change(fill_method=None) * 100
    observed["week_over_week_change"] = change.where(eligible & previous_eligible)
    observed["week_over_week_percent"] = percent.where(eligible & previous_eligible)
    observed["eligible_for_week_comparison"] = eligible
    weekly_rows.append(observed)
weekly_evidence = pd.concat(weekly_rows, ignore_index=True)
weekly_drops = (weekly_evidence[weekly_evidence["week_over_week_change"].lt(0)]
                .sort_values(["floorspaceid", "week_over_week_change"]))

# H3: identify ties explicitly; only complete 10-hour daytime profiles enter peak estimates.
peak_rows = []
for (space, local_date), group in complete_daytime.groupby(["floorspaceid", "local_date"]):
    max_value = group["mean_headcount"].max()
    tied_hours = sorted(group.loc[np.isclose(group["mean_headcount"], max_value), "local_hour"].unique().tolist())
    peak_rows.append({
        "floorspaceid": space,
        "local_date": local_date,
        "phase": group["phase"].iloc[0],
        "peak_headcount_hour_mean": max_value,
        "tied_peak_hours": ",".join(map(str, tied_hours)),
        "noon_is_a_tied_peak": bool(set(tied_hours) & NOON_HOURS),
        "noon_window_mean_headcount": group.loc[group["local_hour"].isin(NOON_HOURS), "mean_headcount"].mean(),
        "daytime_mean_headcount": group["mean_headcount"].mean(),
    })
daily_peaks = pd.DataFrame(peak_rows)

def wilson_interval(successes, trials, z=1.96):
    if trials == 0:
        return (np.nan, np.nan)
    proportion = successes / trials
    denominator = 1 + z * z / trials
    centre = (proportion + z * z / (2 * trials)) / denominator
    radius = z * np.sqrt(proportion * (1 - proportion) / trials + z * z / (4 * trials * trials)) / denominator
    return centre - radius, centre + radius

peak_summary_rows = []
for space, group in daily_peaks.groupby("floorspaceid"):
    n_days = len(group)
    noon_days = int(group["noon_is_a_tied_peak"].sum())
    low, high = wilson_interval(noon_days, n_days)
    peak_hour_counts = (group["tied_peak_hours"].str.split(",").explode().astype(int).value_counts().sort_index())
    peak_summary_rows.append({
        "floorspaceid": space,
        "complete_daytime_profiles": n_days,
        "noon_window_peak_days_11_to_12": noon_days,
        "noon_window_peak_fraction": noon_days / n_days if n_days else np.nan,
        "wilson_95_lower": low,
        "wilson_95_upper": high,
        "most_frequent_peak_hour_among_ties": int(peak_hour_counts.idxmax()) if len(peak_hour_counts) else np.nan,
        "peak_hour_tie_counts": peak_hour_counts.to_dict(),
        "scope": "Peak within observed 08:00-17:59 local hours; not an all-day claim."
    })
peak_summary = pd.DataFrame(peak_summary_rows)

day_coverage_summary = (coverage_by_day.groupby("floorspaceid", as_index=False)
                        .agg(shared_window_days=("local_date", "nunique"),
                             complete_daytime_profiles=("observed_daytime_hours", lambda values: int(values.eq(len(DAYTIME_HOURS)).sum())),
                             median_daytime_hours_observed=("observed_daytime_hours", "median")))

# Persist compact, reproducible evidence tables; raw and preprocessing outputs are never modified.
reports = {
    "occupancy_coverage_and_order_audit.csv": coverage_audit,
    "occupancy_common_window_day_coverage.csv": day_coverage_summary,
    "occupancy_hourly_profile.csv": hourly_evidence,
    "occupancy_semester_phase_summary.csv": phase_summary,
    "occupancy_semester_phase_matched_contrasts.csv": phase_contrasts,
    "occupancy_weekly_headcount.csv": weekly_evidence,
    "occupancy_weekly_drops.csv": weekly_drops,
    "occupancy_daily_peaks.csv": daily_peaks,
    "occupancy_noon_peak_summary.csv": peak_summary,
}
for filename, frame in reports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

print(f"Shared five-space comparison window: {common_start_local} to {common_end_local} ({TIMEZONE}).")
print("Chronological sorting was applied in memory for every analysis; source preprocessing streams were not edited.")
print("Coverage/order audit:")
display(coverage_audit)
print("Complete daytime coverage in shared window:")
display(day_coverage_summary)
print("Semester-phase matched weekday/hour contrasts (descriptive; inspect day counts and matched strata):")
display(phase_contrasts)
print(f"Weeks with >= {MIN_COMPLETE_DAYS_PER_WEEK} complete daytime profiles and valid adjacent comparison; largest drops first:")
display(weekly_drops.groupby("floorspaceid", group_keys=False).head(5))
print("Daily peak evidence (complete 08:00-17:59 local profiles only):")
display(peak_summary)
print(f"Evidence tables saved under {OUTPUT_DIR}.")

Shared five-space comparison window: 2024-05-01 00:00:16+10:00 to 2024-12-30 23:01:56+11:00 (Australia/Melbourne).
Chronological sorting was applied in memory for every analysis; source preprocessing streams were not edited.
Coverage/order audit:


,floorspaceid,rows,coverage_start_local,coverage_end_local,out_of_order_rows_before_sort,out_of_order_rows_after_sort,duplicate_source_ids,duplicate_timestamps,median_positive_interval_seconds,p95_positive_interval_seconds,gaps_over_15_minutes,common_window_start_local,common_window_end_local,analysis_note
0,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,2169461,2023-11-24 00:37:32+11:00,2026-04-22 23:02:18+10:00,1317,0,0,0,4.0,70.0,6764,2024-05-01 00:00:16+10:00,2024-12-30 23:01:56+11:00,All comparative results restricted to shared f...
1,227f1068-9e41-469f-bef0-275f27a0bdba,1500394,2024-03-03 15:06:30+11:00,2026-04-22 22:09:42+10:00,5,0,0,3,4.0,100.0,10631,2024-05-01 00:00:16+10:00,2024-12-30 23:01:56+11:00,All comparative results restricted to shared f...
2,3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8,1551583,2024-05-01 00:00:16+10:00,2024-12-30 23:01:56+11:00,4,0,0,1,2.0,30.0,1747,2024-05-01 00:00:16+10:00,2024-12-30 23:01:56+11:00,All comparative results restricted to shared f...
3,941f0352-bb69-47d0-a9c5-ac29eb4f7319,1975423,2023-11-24 00:36:52+11:00,2026-04-22 23:02:38+10:00,802,0,0,1,4.0,74.0,8137,2024-05-01 00:00:16+10:00,2024-12-30 23:01:56+11:00,All comparative results restricted to shared f...
4,cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b,1894871,2024-03-03 15:06:30+11:00,2026-04-22 22:54:50+10:00,6,0,0,4,4.0,40.0,5257,2024-05-01 00:00:16+10:00,2024-12-30 23:01:56+11:00,All comparative results restricted to shared f...


Complete daytime coverage in shared window:


,floorspaceid,shared_window_days,complete_daytime_profiles,median_daytime_hours_observed
0,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,238,205,10.0
1,227f1068-9e41-469f-bef0-275f27a0bdba,244,197,10.0
2,3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8,244,211,10.0
3,941f0352-bb69-47d0-a9c5-ac29eb4f7319,235,193,10.0
4,cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b,241,118,9.0


Semester-phase matched weekday/hour contrasts (descriptive; inspect day counts and matched strata):


...

Weeks with >= 3 complete daytime profiles and valid adjacent comparison; largest drops first:


,week_start,weekly_mean_headcount,complete_days,floorspaceid,week_start_phase,week_over_week_change,week_over_week_percent,eligible_for_week_comparison
14,2024-08-05,1.845995,7,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,Semester 2 teaching weeks 1-3,-1.805012,-49.43874,True
27,2024-11-04,0.782962,5,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,Semester 2 final assessments,-0.780924,-49.934815,True
2,2024-05-13,1.726709,7,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,Semester 1 teaching week 4+,-0.651038,-27.380455,True
24,2024-10-14,1.429072,7,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,Semester 2 teaching week 4+,-0.645896,-31.127988,True
19,2024-09-09,1.665795,7,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,Semester 2 teaching week 4+,-0.406765,-19.626222,True
38,2024-05-27,1.54988,6,227f1068-9e41-469f-bef0-275f27a0bdba,Semester 1 SWOTVAC,-2.097094,-57.502294,True
51,2024-08-26,2.778521,7,227f1068-9e41-469f-bef0-275f27a0bdba,Semester 2 teaching week 4+,-1.423944,-33.883548,True
59,2024-10-21,2.220327,6,227f1068-9e41-469f-bef0-275f27a0bdba,Semester 2 SWOTVAC,-1.034873,-31.791385,True
58,2024-10-14,3.2552,6,227f1068-9e41-469f-bef0-275f27a0bdba,Semester 2 teaching week 4+,-1.028039,-24.001438,True
44,2024-07-08,1.575384,6,227f1068-9e41-469f-bef0-275f27a0bdba,Outside listed teaching/assessment periods,-0.954253,-37.722933,True


Daily peak evidence (complete 08:00-17:59 local profiles only):


,floorspaceid,complete_daytime_profiles,noon_window_peak_days_11_to_12,noon_window_peak_fraction,wilson_95_lower,wilson_95_upper,most_frequent_peak_hour_among_ties,peak_hour_tie_counts,scope
0,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,205,51,0.248780,0.194587,0.312216,17,"{8: 2, 9: 13, 10: 11, 11: 25, 12: 26, 13: 26, ...",Peak within observed 08:00-17:59 local hours; ...
1,227f1068-9e41-469f-bef0-275f27a0bdba,197,36,0.182741,0.135019,0.242600,13,"{8: 1, 9: 3, 10: 23, 11: 16, 12: 20, 13: 35, 1...",Peak within observed 08:00-17:59 local hours; ...
2,3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8,211,51,0.241706,0.188891,0.303759,17,"{8: 6, 9: 10, 10: 9, 11: 22, 12: 30, 13: 24, 1...",Peak within observed 08:00-17:59 local hours; ...
3,941f0352-bb69-47d0-a9c5-ac29eb4f7319,193,45,0.233161,0.179068,0.297669,15,"{8: 1, 9: 6, 10: 10, 11: 17, 12: 28, 13: 26, 1...",Peak within observed 08:00-17:59 local hours; ...
4,cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b,118,24,0.203390,0.140659,0.284825,13,"{8: 5, 9: 2, 10: 8, 11: 16, 12: 8, 13: 20, 14:...",Peak within observed 08:00-17:59 local hours; ...


Evidence tables saved under C:\Users\joube\OneDrive - Monash University\Year 5 - Semester 2\MMA3001\MMA3001-Project\derived\archive\semester_cycle_occupancy_investigation\reports.


## 4. Final evidence and conclusion (complete after analysis)

For each hypothesis, enter the estimate, uncertainty, coverage, and whether evidence supports, fails to support, or is inconclusive. Distinguish **association**, **prediction**, and **causation**. If a forecast is included, report the chronological test dates and baseline/model metrics side by side. Tie any operational recommendation to the measured effect and state what evidence would be needed before acting.

### Results to report

- Actual sensor data start/end, floorspace coverage, timestamp timezone, cadence findings, missingness, and any exclusions.
- Calendar source and date boundaries; H1 phase contrasts and baseline-versus-phase model validation.
- Weekly occupancy measure, material-drop rule, flagged weeks, and persistence-versus-model validation (or why predictive validation was not feasible).
- H3 noon-window definition, daily peak-time distribution, coverage, and sensitivity results.
- Computational choices, credible alternatives, reproducibility details, engineering interpretation, and limitations.

### Limitations to revisit

Observed headcount may not represent unique people or total facility occupancy. Space purpose/capacity and sensor measurement properties may be unknown. One year's partial record may not capture recurring semester behavior. Calendar phases can coincide with holidays, weather, operating hours, and other unmeasured factors. Temporal autocorrelation reduces effective sample size. These results alone cannot demonstrate that semester events cause occupancy changes.